# 08 · Engine suy luận thời gian thực

Luồng dữ liệu khi chạy với camera:

`frame (fps bất kỳ)` → lấy mẫu theo đồng hồ 7.5 fps → tiền xử lý GPU 224×224 → bộ đệm 16 frame → VideoMAE (fp16) → bộ đệm 18 feature → classifier → `p_t` → **StreamConditioner** (gom pseudo-segment mỗi `segment_hop` giây, điều kiện hoá nhân quả, báo động có trễ).

Độ trễ đo được: 40–47 ms mỗi bước. `StreamConditioner` không cần GPU nên cũng được dùng để mô phỏng streaming offline ở notebook 07.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

# Notebook nằm trong thư mục notebooks/ -> thêm thư mục gốc repo vào sys.path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)

**`tcfall/realtime.py` · phần import và hằng số**

Import và danh sách chế độ: `baseline`, `rule`, `hybrid`, `hmm`.

In [ ]:
from __future__ import annotations
import time
from collections import deque
from dataclasses import dataclass, field
import numpy as np
import torch
from tcfall.config import CLASS_NAMES, FALL, FALLEN, NUM_FEATURES, TARGET_FPS, WINDOW_FRAMES
from tcfall.conditioners import (DEFAULT_HYBRID, hmm_filter, hybrid_apply, load_hybrid, prior_shift,
                                 rule_condition)
MODES = ("baseline", "rule", "hybrid", "hmm")

**`tcfall/realtime.py` · EngineConfig**

Toàn bộ tham số của engine: chế độ, chu kỳ pseudo-segment, tham số luật (`tau`, `window`), ma trận HMM, sửa lệch prior, ngưỡng bật/tắt báo động (1.5 s / 3 s), kích thước batch, và tuỳ chọn TTA không gian (tốn gấp 3 GPU).

In [ ]:
@dataclass
class EngineConfig:
    mode: str = "rule"
    segment_hop: float = 1.0          # s, chu kỳ chốt pseudo-segment cho conditioner
    history_seconds: float = 20.0
    tau: float = 4.0
    window: float = 10.0
    hybrid_ckpt: str = str(DEFAULT_HYBRID)
    hmm_A: np.ndarray | None = None   # ma trận chuyển tiếp (từ outputs_v2/best_config.json)
    hmm_prior: np.ndarray | None = None
    hmm_strength: float = 1.0
    src_prior: np.ndarray | None = None   # hiệu chỉnh label shift (nếu best_config chọn)
    tgt_prior: np.ndarray | None = None
    prior_alpha: float = 0.0
    alarm_on_seconds: float = 1.5     # fallen liên tục >= x giây mới báo động
    alarm_off_seconds: float = 3.0    # hết fallen >= x giây mới tắt báo động
    max_batch: int = 8                # số cửa sổ VideoMAE tối đa xử lý 1 lượt khi bị trễ
    spatial_tta: bool = False         # trung bình 3 crop không gian (trái/giữa/phải) — tốn x3 GPU

**`tcfall/realtime.py` · EngineState**

Trạng thái hiện tại để giao diện hiển thị: thời điểm, xác suất trước/sau điều kiện hoá, nhãn, trạng thái báo động, lịch sử 400 bước, độ trễ.

In [ ]:
@dataclass
class EngineState:
    t: float = 0.0
    baseline_probs: np.ndarray = field(default_factory=lambda: np.full(10, 0.1))
    probs: np.ndarray = field(default_factory=lambda: np.full(10, 0.1))
    label: str = "warmup"
    ready: bool = False
    alarm: bool = False
    alarm_events: list = field(default_factory=list)
    timeline: deque = field(default_factory=lambda: deque(maxlen=400))   # (t, class_id)
    latency_ms: float = 0.0

**`tcfall/realtime.py` · class StreamConditioner**

Khởi tạo bộ điều kiện hoá theo luồng và xoá trạng thái.

In [ ]:
class StreamConditioner:
    """Nhận p_t (xác suất baseline tại thời điểm t) theo dòng, trả xác suất đã điều kiện hoá."""

    def __init__(self, cfg: EngineConfig, hybrid_model=None):
        self.cfg = cfg
        self.hybrid = hybrid_model
        self.reset()

**`tcfall/realtime.py` · StreamConditioner.reset**

Xoá lịch sử: các xác suất đang gom, danh sách pseudo-segment đã chốt, bộ đếm báo động.

In [ ]:
def reset(self):
    self.step_probs: list[np.ndarray] = []
    self.segments: list[tuple[float, np.ndarray]] = []
    self.last_seg_t = None
    self.fallen_since = None
    self.clear_since = None
    self.state = EngineState()


StreamConditioner.reset = reset  # gắn method vào class đã khai báo ở cell trước

**`tcfall/realtime.py` · StreamConditioner.update**

Nhận `p_t` mỗi bước:
1. (tuỳ chọn) sửa lệch prior;
2. cộng dồn vào pseudo-segment đang hình thành;
3. đủ `segment_hop` giây thì **chốt** segment vào lịch sử (giữ 20 s gần nhất) và điều kiện hoá lại;
4. chưa đủ thì vẫn cập nhật tức thời bằng lịch sử + segment tạm;
5. cập nhật nhãn, timeline và báo động.

Lý do gom pseudo-segment: các bộ điều kiện hoá được thiết kế cho chuỗi đoạn cách nhau khoảng 1.5 s; đưa từng bước 0.13 s vào sẽ lệch phân phối.

In [ ]:
def update(self, t: float, p: np.ndarray) -> EngineState:
    c, st = self.cfg, self.state
    if c.prior_alpha > 0 and c.src_prior is not None:
        p = prior_shift(p[None], c.src_prior, c.tgt_prior, c.prior_alpha)[0]
    st.t, st.baseline_probs, st.ready = t, p, True
    self.step_probs.append(p)
    if self.last_seg_t is None:
        self.last_seg_t = t
    seg_p = np.mean(self.step_probs, 0)
    if t - self.last_seg_t >= c.segment_hop:
        # chốt pseudo-segment hiện tại vào lịch sử
        self.segments.append((self.last_seg_t, seg_p))
        self.segments = [s for s in self.segments if t - s[0] <= c.history_seconds]
        self.step_probs, self.last_seg_t = [], t
        st.probs = self._condition(self.segments)
    elif c.mode == "baseline":
        st.probs = p
    else:
        # cập nhật tức thời: lịch sử đã chốt + segment tạm (chưa chốt) đang hình thành
        st.probs = self._condition(self.segments + [(self.last_seg_t, seg_p)])
    k = int(st.probs.argmax())
    st.label = CLASS_NAMES[k]
    st.timeline.append((t, k))
    self._update_alarm(t, k)
    return st


StreamConditioner.update = update  # gắn method vào class đã khai báo ở cell trước

**`tcfall/realtime.py` · StreamConditioner._condition**

Chọn bộ điều kiện hoá theo chế độ và trả xác suất của **đoạn mới nhất**.

In [ ]:
def _condition(self, segments) -> np.ndarray:
    starts = np.array([s for s, _ in segments])
    P = np.stack([p for _, p in segments])
    mode = self.cfg.mode
    if mode == "baseline":
        return P[-1]
    if mode == "hmm" and self.cfg.hmm_A is not None:
        return hmm_filter(P, self.cfg.hmm_A, self.cfg.hmm_prior, self.cfg.hmm_strength)[-1]
    R = rule_condition(P, starts, self.cfg.tau, self.cfg.window)
    if mode == "hybrid" and self.hybrid is not None:
        return hybrid_apply(self.hybrid, R, starts)[-1]
    return R[-1]


StreamConditioner._condition = _condition  # gắn method vào class đã khai báo ở cell trước

**`tcfall/realtime.py` · StreamConditioner._update_alarm**

Báo động có trễ: bật khi `fallen` kéo dài ≥ `alarm_on_seconds`, tắt khi hết `fallen` ≥ `alarm_off_seconds` — tránh nhấp nháy báo động.

In [ ]:
def _update_alarm(self, t: float, k: int):
    st, c = self.state, self.cfg
    if k == FALLEN:
        self.clear_since = None
        if self.fallen_since is None:
            self.fallen_since = t
        if not st.alarm and t - self.fallen_since >= c.alarm_on_seconds:
            st.alarm = True
            st.alarm_events.append(t)
    else:
        self.fallen_since = None
        if st.alarm:
            if self.clear_since is None:
                self.clear_since = t
            if t - self.clear_since >= c.alarm_off_seconds:
                st.alarm = False


StreamConditioner._update_alarm = _update_alarm  # gắn method vào class đã khai báo ở cell trước

**Ví dụ nhỏ:** mô phỏng 12 giây luồng xác suất (walk → fall → lying/fallen) qua `StreamConditioner` chế độ `rule`, không cần GPU.

In [ ]:
import numpy as np
def onehot_like(k, conf=0.7):
    p = np.full(10, (1 - conf) / 9); p[k] = conf; return p
sc = StreamConditioner(EngineConfig(mode="rule", segment_hop=1.0), hybrid_model=None)
for i in range(90):                       # 90 bước x 0.133 s ≈ 12 s
    t = i / 7.5
    if t < 4: p = onehot_like(0)                          # walk
    elif t < 5.5: p = onehot_like(1)                      # fall
    else: p = 0.5 * onehot_like(6, 0.5) + 0.5 * onehot_like(2, 0.45)  # phân vân lying/fallen
    st = sc.update(t, p)
    if i % 8 == 0:
        print(f"t={t:5.2f}s  nhãn={st.label:8s}  báo động={st.alarm}")

**`tcfall/realtime.py` · class FallEngine**

Ghép toàn bộ: nạp VideoMAE (fp16 trên GPU), classifier baseline và `StreamConditioner`.

In [ ]:
class FallEngine:
    """Ghép VideoMAE + baseline + StreamConditioner cho đầu vào là frame thô."""

    def __init__(self, cfg: EngineConfig, device: str | None = None):
        from transformers import VideoMAEModel

        from tcfall.baseline import load_baseline
        from tcfall.config import VIDEOMAE_NAME
        self.cfg = cfg
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.fp16 = self.device == "cuda"
        vm = VideoMAEModel.from_pretrained(VIDEOMAE_NAME).to(self.device).eval()
        self.videomae = vm.half() if self.fp16 else vm
        self.baseline = load_baseline(self.device)
        self.cond = StreamConditioner(cfg, load_hybrid(cfg.hybrid_ckpt, "cpu"))
        self.reset()

**`tcfall/realtime.py` · FallEngine.crops**

Danh sách crop không gian: chỉ `center`, hoặc 3 crop nếu bật TTA không gian.

In [ ]:
def crops(self) -> tuple[str, ...]:
    return ("center", "start", "end") if self.cfg.spatial_tta else ("center",)


FallEngine.crops = property(crops)  # gắn method vào class đã khai báo ở cell trước

**`tcfall/realtime.py` · FallEngine.reset**

Tạo các bộ đệm: 16 frame đã tiền xử lý, các cửa sổ chờ đưa vào VideoMAE, 18 feature gần nhất.

In [ ]:
def reset(self):
    self.next_sample_t = 0.0
    self.frames = deque(maxlen=WINDOW_FRAMES)            # tensor (K,3,224,224) trên GPU, K = số crop
    self.pending: list[tuple[float, torch.Tensor]] = []  # (t, cửa sổ (K,16,3,224,224)) chờ embed
    self.feats = deque(maxlen=NUM_FEATURES)              # tensor (K, 768)
    self.cond.reset()


FallEngine.reset = reset  # gắn method vào class đã khai báo ở cell trước

**`tcfall/realtime.py` · FallEngine.state**

Truy cập nhanh trạng thái hiện tại của `StreamConditioner`.

In [ ]:
def state(self) -> EngineState:
    return self.cond.state


FallEngine.state = property(state)  # gắn method vào class đã khai báo ở cell trước

**`tcfall/realtime.py` · FallEngine.push_frame**

Nhận frame từ camera kèm thời điểm. Chỉ giữ frame khi tới mốc lấy mẫu 7.5 fps tiếp theo; nếu nguồn bị giật thì **bỏ mốc đã lỡ** thay vì dồn frame.
Frame được tiền xử lý trên GPU; đủ 16 frame thì tạo một cửa sổ chờ. Nếu GPU không kịp, chỉ giữ các cửa sổ mới nhất để không trễ dồn.

In [ ]:
def push_frame(self, frame_rgb: np.ndarray, t: float) -> bool:
    """Đưa 1 frame với mốc thời gian t (giây). Trả về True nếu frame được lấy mẫu."""
    from tcfall.preprocess_gpu import preprocess_frames
    if t + 1e-6 < self.next_sample_t:
        return False
    while self.next_sample_t <= t:   # nguồn bị giật: bỏ các mốc đã lỡ thay vì dồn frame
        self.next_sample_t += 1.0 / TARGET_FPS
    px = preprocess_frames([frame_rgb], self.device, self.fp16, crops=self.crops)
    self.frames.append(px[:, 0] if px.dim() == 5 else px)      # (K,3,224,224)
    if len(self.frames) == WINDOW_FRAMES:
        self.pending.append((t, torch.stack(list(self.frames), 1)))  # (K,16,3,224,224)
        if len(self.pending) > 4 * self.cfg.max_batch:   # GPU không kịp: giữ cửa sổ mới nhất
            self.pending = self.pending[-self.cfg.max_batch:]
    return True


FallEngine.push_frame = push_frame  # gắn method vào class đã khai báo ở cell trước

**`tcfall/realtime.py` · FallEngine.step**

Gom tối đa `max_batch` cửa sổ chờ, chạy VideoMAE một lượt, đẩy feature vào bộ đệm 18; đủ 18 thì chạy classifier (trung bình các crop) và cập nhật `StreamConditioner`. Ghi lại độ trễ mỗi cửa sổ.

In [ ]:
@torch.no_grad()
def step(self) -> EngineState:
    """Xử lý các cửa sổ đang chờ (gom batch). Gọi liên tục từ luồng suy luận."""
    if not self.pending:
        return self.state
    t0 = time.perf_counter()
    batch, self.pending = self.pending[: self.cfg.max_batch], self.pending[self.cfg.max_batch:]
    x = torch.cat([w for _, w in batch])                    # (B*K,16,3,224,224)
    k = len(self.crops)
    f = self.videomae(pixel_values=x).last_hidden_state.mean(1).float().view(len(batch), k, -1)
    for (t, _), feat in zip(batch, f):
        self.feats.append(feat)                             # (K,768)
        if len(self.feats) == NUM_FEATURES:
            seq = torch.stack(list(self.feats), 1)          # (K,18,768)
            probs = torch.softmax(self.baseline(seq)["logits"].float(), -1).mean(0)
            self.cond.update(t, probs.cpu().numpy())
    self.state.latency_ms = (time.perf_counter() - t0) * 1000 / len(batch)
    return self.state


FallEngine.step = step  # gắn method vào class đã khai báo ở cell trước

**`tcfall/realtime.py` · FallEngine.status**

Trạng thái gọn cho giao diện: `WARMUP`, `ALARM`, `FALLEN`, `FALL` hoặc `NORMAL`.

In [ ]:
def status(self) -> str:
    st = self.state
    if not st.ready:
        return "WARMUP"
    if st.alarm:
        return "ALARM"
    k = int(st.probs.argmax())
    return "FALLEN" if k == FALLEN else "FALL" if k == FALL else "NORMAL"


FallEngine.status = property(status)  # gắn method vào class đã khai báo ở cell trước